# SUNNY SHABAN ALI - 22K4149 - 6B - ASSIGNMENT 1

##### GitHub Repository for the project: https://www.github.com/sunnyallana/boolean-retrieval-model-pipeline


## References:
- 1. NLTK PorterStemmer: https://www.nltk.org/api/nltk.stem.porter.html
- 2. Python Regular Expressions: https://docs.python.org/3/library/re.html
- 3. Pickle Module: https://docs.python.org/3/library/pickle.html

<br/>

##### <strong>Kindly ensure that you have the Abstracts directory and Stopword-List.txt in the same folder as this code.<strong/>



In [ ]:
import os
import re
from nltk.stem import PorterStemmer as Stem
import pickle

# Clean up text - remove punctuation, lowercase everything, and stem terms
def clean_text(text, stop_words, stemmer):
    # Convert to lowercase and split into words (removing punctuation)
    all_words = re.findall(r'\w+', text.lower())

    # Remove stop words
    meaningful_words = []
    for word in all_words:
        if word not in stop_words:
            meaningful_words.append(word)

    # Stem each word (convert to root form)
    stemmed_words = []
    for word in meaningful_words:
        root_word = stemmer.stem(word)
        stemmed_words.append(root_word)

    return stemmed_words

# Load our stop words file
def load_stops(stop_file):
    stop_words_set = set()

    # Open and read the file
    with open(stop_file, 'r', encoding='latin-1') as file:
        for line in file:
            word = line.strip()  # Remove space
            stop_words_set.add(word)
    return stop_words_set

# Create search indexes from the abstract files
def create_indexes(abs_folder, stop_words):
    stemmer = Stem()
    inverted_idx = {}
    pos_index = {}
    doc_ids = set()

    print(f"Processing files")
    file_count = 0

    # Go through each abstract file
    for document_name in os.listdir(abs_folder):
        if document_name.endswith('.txt'):
            file_count += 1
            doc_num = document_name.split('.')[0]
            doc_ids.add(doc_num)

            # Dealing with encoding issues in the files - Latin-1 worked, so I placed that here
            try:
                with open(os.path.join(abs_folder, document_name), 'r', encoding='latin-1') as document:
                    content = document.read()
                    words = clean_text(content, stop_words, stemmer)

                    # Add each word to our indexes
                    word_positions = {}
                    for idx, word in enumerate(words):
                        # Update inverted index
                        if word not in inverted_idx:
                            inverted_idx[word] = set()
                        inverted_idx[word].add(doc_num)

                        # Update positional index
                        if word not in word_positions:
                            word_positions[word] = []
                        word_positions[word].append(idx)

                    # Add all positions to master index
                    for word, positions in word_positions.items():
                        if word not in pos_index:
                            pos_index[word] = {}
                        pos_index[word][doc_num] = positions
            except Exception as e:
                print(f"Error with file {document_name}: {e}")

    print(f"Processed {file_count} files in count")
    return inverted_idx, pos_index, doc_ids

# Save our indexes to disk to avoid rebuilding
def store_indexes(inverted_idx, pos_index, filename='search_indexes.pkl'):
    with open(filename, 'wb') as indexes:
        pickle.dump((inverted_idx, pos_index), indexes)

# Load previously built indexes
def load_idx_file(filename='search_indexes.pkl'):
    with open(filename, 'rb') as indexes:
        inverted_idx, pos_index = pickle.load(indexes)

    # Convert to dictionary with empty defaults for easier querying
    inv_idx_modified = {}
    for k, v in inverted_idx.items():
        inv_idx_modified[k] = set(v)

    pos_idx_modified = {}
    for term, docs in pos_index.items():
        pos_idx_modified[term] = {}
        for doc_id, positions in docs.items():
            pos_idx_modified[term][doc_id] = positions

    return inv_idx_modified, pos_idx_modified

# Handle boolean search queries (AND, OR, NOT)
def run_boolean_search(query, inverted_idx, all_docs, stop_words):
    stemmer = Stem()
    parts = query.split()
    results = None
    last_op = None
    i = 0

    while i < len(parts):
        token = parts[i]

        # Handle NOT operator
        if token == "NOT":
            if i + 1 >= len(parts):
                return set()

            next_word = clean_text(parts[i+1], stop_words, stemmer)
            if not next_word:
                matching_docs = set()
            else:
                word = next_word[0]
                matching_docs = inverted_idx.get(word, set())

            excluded_docs = all_docs - matching_docs

            if results is None:
                results = excluded_docs
            else:
                if last_op == "AND":
                    results &= excluded_docs
                elif last_op == "OR":
                    results |= excluded_docs
                last_op = None
            i += 2

        # Store boolean operator for next iteration
        elif token in ["AND", "OR"]:
            last_op = token
            i += 1

        # Handle normal word
        else:
            clean_words = clean_text(token, stop_words, stemmer)
            if not clean_words:
                matching_docs = set()
            else:
                word = clean_words[0]
                matching_docs = inverted_idx.get(word, set())

            if results is None:
                results = matching_docs
            else:
                if last_op == "AND":
                    results &= matching_docs
                elif last_op == "OR":
                    results |= matching_docs
                last_op = None
            i += 1

    return results if results is not None else set()

# Handle proximity queries (word1 word2 /k)
def run_proximity_search(query, inverted_idx, pos_index, stop_words):
    stemmer = Stem()
    parts = query.split()

    for i, part in enumerate(parts):
        if '/' in part:
            # Need at least 2 words before the distance specifier
            if i < 2:
                return set()

            first_term = parts[i-2]
            second_term = parts[i-1]

            # Get the distance number
            try:
                distance = int(part.split('/')[1])
            except (ValueError, IndexError):
                print("Invalid proximity format - use 'word1 word2 /k'")
                return set()

            # Stem and clean the search terms
            first_clean = clean_text(first_term, stop_words, stemmer)
            second_clean = clean_text(second_term, stop_words, stemmer)

            if not first_clean or not second_clean:
                return set()

            word1 = first_clean[0]
            word2 = second_clean[0]

            # Find docs that have both words
            word1_docs = inverted_idx.get(word1, set())
            word2_docs = inverted_idx.get(word2, set())
            shared_docs = word1_docs.intersection(word2_docs)
            matches = set()

            # Check if words are close enough in each doc
            for doc in shared_docs:
                positions1 = pos_index.get(word1, {}).get(doc, [])
                positions2 = pos_index.get(word2, {}).get(doc, [])

                found_match = False
                for pos1 in positions1:
                    for pos2 in positions2:
                        if abs(pos1 - pos2) - 1 <= distance:
                            matches.add(doc)
                            found_match = True
                            break
                    if found_match:
                        break

            return matches

    return set()

def main():
    # Paths to given files in the assignment
    abstracts_path = 'Abstracts'
    stopwords_path = 'Stopword-List.txt'
    index_path = 'search_indexes.pkl'

    # Try to load saved indexes first
    if os.path.exists(index_path):
        try:
            inverted_idx, pos_index = load_idx_file(index_path)

            # Reconstruct all_docs from inverted index
            all_docs = set()
            for term_docs in inverted_idx.values():
                all_docs.update(term_docs)

            if not all_docs:
                raise ValueError("Empty indexes")

        except Exception as e:
            # Delete corrupted file
            try:
                os.remove(index_path)
            except:
                pass

            # Build fresh indexes
            stop_words = load_stops(stopwords_path)
            inverted_idx, pos_index, all_docs = create_indexes(abstracts_path, stop_words)
            store_indexes(inverted_idx, pos_index, index_path)
    else:
        # No saved indexes found, build from scratch
        stop_words = load_stops(stopwords_path)
        inverted_idx, pos_index, all_docs = create_indexes(abstracts_path, stop_words)

        if not all_docs:
            print(f"Error: No documents found!")
            return

        store_indexes(inverted_idx, pos_index, index_path)

    # Command line search interface
    while True:
        query = input("\nEnter search query (or 'exit' to quit): ").strip() # Getting rid of spaces from the query

        if query.lower() == 'exit': # To get out of initiated infinite loop of retrieving queries from user
            break

        # Getting rid of space after /
        if '/ ' in query:
            query = query.replace('/ ', '/')

        # Detect query type and run appropriate search
        if '/' in query:
            # Proximity search
            results = run_proximity_search(query, inverted_idx, pos_index, load_stops(stopwords_path))
        else:
            # Boolean search
            results = run_boolean_search(query, inverted_idx, all_docs, load_stops(stopwords_path))

        # Display results
        if results:
            print(f"Found {len(results)} matching documents:")
            print("Result-Set:", ', '.join(sorted(results, key=int)))
        else:
            print("No matching documents found.")

if __name__ == "__main__":
    main()


Enter search query (or 'exit' to quit): deep and learning
Found 50 matching documents:
Result-Set: 21, 23, 24, 174, 175, 176, 177, 213, 245, 246, 247, 250, 254, 258, 267, 272, 273, 278, 279, 280, 281, 325, 333, 345, 346, 347, 348, 352, 357, 358, 360, 362, 371, 373, 374, 375, 376, 380, 381, 382, 396, 397, 398, 401, 404, 405, 415, 421, 432, 444
